[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Digital-AI-Finance/Introduction-to-Machine-Learning-notebooks/blob/master/deepseek_minimal.ipynb)

# DeepSeek in three lines

A key, one call, one answer. Run each cell with **Shift and Enter**.

The key is typed into the box the first cell opens and is never written into
the notebook. Keys are issued at platform.deepseek.com, and every cell below
costs a fraction of a cent.

In [ ]:
%pip install --quiet openai

import getpass
import math
from openai import OpenAI

client = OpenAI(api_key=getpass.getpass("DeepSeek API key: "), base_url="https://api.deepseek.com")

In [ ]:
answer = client.chat.completions.create(model="deepseek-flash", messages=[{"role": "user", "content": "In one sentence: what does a language model compute?"}])

print(answer.choices[0].message.content)

One line sends a message and one line prints what came back. `deepseek-flash`
is DeepSeek-V4.1-Flash, which reads up to a million tokens at once and writes
up to 384,000; `deepseek-v4-pro` is the larger model and the same call reaches
it by name.

The surface is the one OpenAI publishes, so the only two things that make this
DeepSeek are the `base_url` and the model name.

## The next word, and what it nearly said

In [ ]:
step = client.chat.completions.create(model="deepseek-flash", messages=[{"role": "user", "content": "Complete with one word only: the united"}], max_tokens=1, logprobs=True, top_logprobs=8)

for alt in step.choices[0].logprobs.content[0].top_logprobs:
    print("%-16r %.4f" % (alt.token, math.exp(alt.logprob)))

`max_tokens=1` stops the model after one token, and `logprobs` asks for the
eight it weighed. What prints is a distribution over the vocabulary: the token
it took, and the ones it passed over, each with the share of the probability
it held.

`top_logprobs` runs from 0 to 20. A token outside the top twenty comes back as
-9999.0, which marks it as very unlikely and not as a probability.

## The same question, six times

In [ ]:
for t in (0.0, 1.5):
    for run in range(3):
        out = client.chat.completions.create(model="deepseek-flash", temperature=t, messages=[{"role": "user", "content": "Name one European city. One word, nothing else."}])
        print(t, out.choices[0].message.content)

`temperature` divides the scores before they become probabilities. At 0 the
model takes the leading token every time and the three answers agree. At 1.5
the weight spreads down the list and the three answers separate.

Nothing about the model changed between the two blocks of three. The same
parameters, the same prompt, and a different answer: the output is a draw.

## What it did before it answered

In [ ]:
think = client.chat.completions.create(model="deepseek-flash", messages=[{"role": "user", "content": "A bat and a ball cost 1.10 together. The bat costs 1.00 more than the ball. What does the ball cost?"}], reasoning_effort="high", extra_body={"thinking": {"type": "enabled"}})

print(think.choices[0].message.reasoning_content[:600])
print("----")
print(think.choices[0].message.content)

Thinking mode returns `reasoning_content` beside `content`: the tokens the
model wrote to itself before the ones it wrote to you. Both are billed as
output.

`reasoning_effort` takes `low`, `medium` or `high` and buys more of those
tokens. Thinking mode ignores `logprobs` and `top_logprobs`, so the previous
cell and this one cannot be combined.

## What the call cost

In [ ]:
used = think.usage

print("tokens in:", used.prompt_tokens, " tokens out:", used.completion_tokens)
print("US cents: %.4f" % ((used.prompt_tokens * 0.15 + used.completion_tokens * 0.60) / 1e6 * 100))

Every response carries the count of what it read and what it wrote. Off peak,
`deepseek-flash` is priced at 0.15 US dollars per million input tokens that
miss the cache and 0.60 per million output tokens; a cached input token is
0.003. Peak hours, 01:00 to 04:00 and 06:00 to 10:00 UTC on weekdays, double
both. `deepseek-v4-pro` is 0.66 and 1.98 off peak. Prices read from
api-docs.deepseek.com on 21 September 2026.

The reasoning tokens are in `completion_tokens`, which is why a thinking
answer of two sentences bills like an essay.

## What you can say now

- A call is a message list, a model name and a key. The answer arrives as one
  string, with the counts that priced it.
- Under the string is a distribution: `logprobs` shows the tokens the model
  weighed against the one it chose.
- `temperature` decides how far down that list the draw can fall, and two runs
  of the same prompt are two draws.
- Thinking mode returns the working as `reasoning_content`, billed with the
  output and unavailable together with `logprobs`.

The ninety minute deck **Large Language Models** takes the same four things to
the geometry underneath them.